# 03 — Phase 2 eval + ablations (Amazon Reviews 2023, Electronics, balanced)
Runs text_only / plus_metadata / plus_image / plus_rag_reviews / plus_rag_specs / full_graph, plus a norm-vs-stdev dissonance ablation, on a per-rating-balanced slice.

In [ ]:
!rm -rf /content/repo
!git clone https://github.com/kvarun314/pes-mtech-project.git /content/repo
%cd /content/repo
!pip install -q -e ".[colab]"
!pip install -q "bitsandbytes>=0.43.0"

In [ ]:
import os
from huggingface_hub import login

# Colab Secrets panel (recommended): add a secret named HF_TOKEN, then
# grant this notebook access to it (toggle in the Secrets panel) — it
# reaches this cell via google.colab.userdata, NOT via os.environ.
# Accept the Llama-3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B
# before this token will work.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")  # raises if access wasn't granted — fix that, don't swallow it
except ImportError:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is empty. Set it in Colab Secrets (left sidebar) and grant this "
        "notebook access, or set os.environ['HF_TOKEN'] directly."
    )
login(hf_token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/pes-mtech-project/results/run2"
import os; os.makedirs(RESULTS_DIR, exist_ok=True)

## Load Amazon Reviews 2023 — Electronics

In [ ]:
from datasets import load_dataset

reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Electronics",
                       split="full", trust_remote_code=False)
meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Electronics",
                    split="full", trust_remote_code=False)
meta_by_asin = {m["parent_asin"]: m for m in meta.select(range(min(len(meta), 200000)))}

In [ ]:
from agentic_sentiment.data.amazon2023 import load_balanced_slice, build_spec_records

rows = load_balanced_slice(list(reviews.select(range(min(len(reviews), 500000)))),
                            meta_by_asin, n_per_rating=400, seed=42)
# load_balanced_slice keys its star rating as "rating"; run_ablation/compute_metrics
# (agentic_sentiment.eval.run_eval) read ground truth from "gt_rating" — copy it over.
for row in rows:
    row["gt_rating"] = row["rating"]
print(len(rows), "balanced eval rows")
spec_records = build_spec_records(meta_by_asin)

## BLIP captions (before loading LLaMA, to save VRAM)

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
import requests
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")
for row in rows:
    row["image_caption"] = None
    if row.get("image_url"):
        try:
            img = Image.open(requests.get(row["image_url"], timeout=5, stream=True).raw).convert("RGB")
            inputs = processor(img, return_tensors="pt").to("cuda")
            row["image_caption"] = processor.decode(blip.generate(**inputs)[0], skip_special_tokens=True)
        except Exception:
            pass
del blip
torch.cuda.empty_cache()

## Build spec store (CLIP text embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer
from agentic_sentiment.rag.spec_store import SpecStore

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embed_fn = lambda text: embedder.encode(text).tolist()

spec_store = SpecStore(db_path="/content/spec_lancedb", embed_fn=embed_fn)
spec_store.build(spec_records)

## Load Phase 1 LoRA adapter from Task 12's run_dir

In [ ]:
import glob
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Durable across runtimes: read from Drive, written by colab/01_train_phase1.ipynb
# (this notebook never assumes a bare /content/... path from another notebook).
adapter_dir = sorted(glob.glob("/content/drive/MyDrive/pes-mtech-project/runs/*/best_adapter"))[-1]
print("Using adapter:", adapter_dir)
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
base = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", load_in_4bit=True, device_map="auto")
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

def llm_fn(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

## Run every ablation (each resumable via its own checkpoint)

In [ ]:
from agentic_sentiment.eval.run_eval import ABLATIONS, run_ablation, compute_metrics
import json

all_metrics = {}
for name in ABLATIONS:
    store = spec_store if name == "plus_rag_specs" or name == "full_graph" else None
    ckpt = f"{RESULTS_DIR}/{name}_checkpoint.jsonl"
    run_ablation(name, rows, llm_fn=llm_fn, checkpoint_path=ckpt, spec_store=store)
    records = [json.loads(l) for l in open(ckpt)]
    all_metrics[name] = compute_metrics(records)
    print(name, all_metrics[name])

with open(f"{RESULTS_DIR}/metrics_summary.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

## Dissonance-formula ablation (norm vs. Run 1 stdev)

In [ ]:
for method in ("norm", "stdev"):
    ckpt = f"{RESULTS_DIR}/dissonance_{method}_checkpoint.jsonl"
    run_ablation("full_graph", rows, llm_fn=llm_fn, checkpoint_path=ckpt,
                 spec_store=spec_store, dissonance_method=method)
    records = [json.loads(l) for l in open(ckpt)]
    print(method, compute_metrics(records))